In [ ]:
import pandas as pd 
import polars as pl 
from pathlib import Path
from sqlalchemy import create_engine, text
from openhexa.sdk import workspace, current_run
from datetime import datetime

from sqlalchemy.engine import Engine

import time
import gc

from logistics_helper_functions import format_time_period, pct_sans_rupture_fosa, run_logistics_nd_aggregations_lvl5
from pnlp_utils import log_msg, push_dataframe_to_db, check_partition_exist, upsert_partition, create_partition_indexes

In [ ]:
# Parameters
YEAR = 2026

In [ ]:
coc_selection = [
    "Nbre de jours de rupture de stock",
]

columns_expected = [    
    'aggregation_temporelle',
    'period',
    'annee',    
    'indicateur_name',
    'de_name',
    'province',
    'zone_de_sante',
    'aire_de_sante',
    'fosa',
    "fosa_dhis2_id",
    'absolute_indicateur_value',
    'taux_indicateur_value',
    'indicateur_num',
    'indicateur_den'
]


### 4.14.1. % des FOSA n'ayant pas eu de rupture de stocks

Nombre de FOSA par aire de santé/zone de santé / etc qui n'ont connus aucun jour de rupture de stock (Nbr de jour de rupture =0) / Nbr total de FOSA par aire de santé / zone de santé / etc

### Start processing

In [ ]:
# table names
target_table = "logistics_ND_agg_rupture_fosa"

# Database Connection 
engine = create_engine(workspace.database_url)
print(f"Connected to: {engine.url.host}")

# Load available years
target_years = [YEAR] 
# target_years = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
print(f"Running for years: {target_years}")

In [ ]:
for target_year in target_years:
    
    log_msg(f"***Running for year: {target_year}")     
    try:
        logistics_path = Path("~/workspace/pnlp-tdb-pipeline/data/interim") / f"PNLP_logistics_{target_year}.feather"
        logistics_level_5 = pl.read_ipc(logistics_path, memory_map=False)
        logistics_level_5 = logistics_level_5.filter((pl.col("COC_NAME").is_in(coc_selection)))
    except Exception as e:
        current_run.log_info(f"Error loading {logistics_path}. No data found, year skipped.")
        continue
    
    # Format -> Following original R scripts
    df_logistics_mensuel = format_time_period(logistics_level_5)  

    # 4.14.1. % des FOSA n'ayant pas eu de rupture de stocks 
    log_msg(f"Computing logistics rupture FOSA..")
    df_rupture = pct_sans_rupture_fosa(df=df_logistics_mensuel, indicateur_name="Pct_FOSA_Sans_Rupture")
    df_rupture_act = pct_sans_rupture_fosa(df=df_logistics_mensuel, indicateur_name="Pct_FOSA_Sans_Rupture_ACT")
    df_rupture_tout_prod = pct_sans_rupture_fosa(df=df_logistics_mensuel, indicateur_name="Pct_FOSA_Sans_Rupture_Tous_Produits")
    # Concat
    df_rupture_ind = pl.concat([df_rupture, df_rupture_act, df_rupture_tout_prod])

    # clean
    del logistics_level_5, df_logistics_mensuel, df_rupture, df_rupture_act, df_rupture_tout_prod
    collected = gc.collect()

    # Run temporal aggregations
    df_year_agg = run_logistics_nd_aggregations_lvl5(df_rupture_ind)
    assert df_year_agg.height == df_year_agg.unique().height, "Duplicated values found!" # quick check!
    
    print("Check expected columns..", end=" ")
    df_year_agg = df_year_agg.select(columns_expected)
    print("OK!")
            
    log_msg(f"Pushing logistics rupture FOSA to db. Target table: {target_table}")
    tmp_table = f"{target_table}_tmp" 
    push_dataframe_to_db(
        df=df_year_agg, 
        table_name=tmp_table, 
        engine=engine, 
        push_behaviour="replace",
        chunksize=500000
    )
    
    # update partition
    upsert_partition(engine, target_table, tmp_table, target_year)
    
    log_msg("Creating indexes..")
    index_columns = ["aggregation_temporelle", "province", "zone_de_sante", "aire_de_sante", "fosa", "period"]
    create_partition_indexes(engine, target_table, target_year, index_columns)
    
    # clean
    del df_year_agg, df_rupture_ind, index_columns
    collected = gc.collect()
    print(collected)
